# 01 House Prices - Exploratory Data Analysis (EDA)

This notebook provides a formal audit of the structural issues in the `house_prices.csv` dataset. We visualize why the raw data is unfit for regression without specific cleaning steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

sns.set(style='whitegrid', palette='muted')
%matplotlib inline

### Step 1: Loading & Deduplication
**Action**: Load the data and remove the 150 intentional duplicate records.
**Why?**: Duplicates can artificially inflate model performance metrics (overfitting). Removing them ensures our evaluation is realistic.

In [ ]:
df = pd.read_csv('../_data/house_prices.csv')
df = df.drop_duplicates().reset_index(drop=True)
print(f"Unique records: {len(df)}")

### Step 2: Unit Mismatch Analysis
**Action**: Categorize `Lot_Size` by unit (Sqft vs Acres) and plot the raw values.
**Why?**: As shown below, Acre values are near 1.0 while Sqft values are in the thousands. A model would ignore the Acre properties if we don't normalize them to a single unit (Sqft).

In [ ]:
df['lot_unit'] = df['Lot_Size'].apply(lambda x: 'Acres' if 'ac' in str(x).lower() else 'Sqft')
df['raw_val'] = df['Lot_Size'].apply(lambda x: float(re.sub(r'[^\d\.]', '', str(x))) if str(x) != 'nan' else 0)

plt.figure(figsize=(10, 5))
sns.stripplot(data=df, x='lot_unit', y='raw_val', alpha=0.3)
plt.yscale('log')
plt.title('Scale Gap: Acres vs Sqft (Raw Values)')
plt.show()

### Step 3: Missing Value Heatmap
**Action**: Highlight the null distribution.
**Why?**: `Renovation_Year` is ~79% missing. This tells us we should use it as a binary indicator (`Was_Renovated`) rather than a numerical year feature.

In [ ]:
plt.figure(figsize=(12, 4))
sns.heatmap(df.isna(), cbar=False, cmap='viridis')
plt.title('Missingness Map (Yellow = Null)')
plt.show()

### Step 4: Mixed-Type Audit
**Action**: Plot the frequency of `Overall_Condition` labels.
**Why?**: We see a mix of numbers and strings like `Excellent`. This visual proof confirms we need an ordinal mapping function to convert text to numbers.

In [ ]:
plt.figure(figsize=(10, 4))
df['Overall_Condition'].value_counts().plot(kind='bar', color='skyblue')
plt.title('Condition Column: Mixture of Text and Numbers')
plt.show()

## Final Preprocessing Roadmap
1. **Drop Target NaNs**: Clean the 57 missing prices.
2. **Normalize Units**: Convert Acres to Sqft.
3. **Map Ordinals**: Convert 'Excellent' etc. to 1-10.
4. **Cap Outliers**: Limit impact of 10x price/area spikes.